# Controllable Scenario PRB Visualization

Four Jain-controllable training scenarios:

| Scenario | Slices | Target load (gNB-1) | UEs |
|---|---|---|---|
| `jain_balance_controllable` | eMBB | 0.90 | 6 |
| `jain_control_urllc` | URLLC | 0.72 | 6 |
| `jain_control_mmtc` | mMTC | 0.60 | 6 |
| `jain_control_mixed` | eMBB + URLLC + mMTC | 0.90 | 18 |

All UEs start on **gNB-1 (center)** at the equidistant midpoint between gNB-1 and the outer gNBs (±135 m).  
RSRP difference = 0 dB → any bias < −0.17 fires A3 and triggers handovers to both outer gNBs.

**Carrier configuration (this notebook):** Each gNB is assigned a unique `carrier_id` (0/1/2) so `_same_carrier_interferers` is empty → SINR = SNR (noise-limited). A3 RSRP comparison is carrier-agnostic so handovers still fire normally. This isolates the load-balancing effect from co-channel interference effects.

**Three states measured per scenario:**
1. **Before** — after 4 warmup steps (calibration converged), zero action, all UEs on gNB-1
2. **Post-HO transient** — measurement window right after the negative-bias step fires the handovers (calibration window not yet complete)
3. **Settled** — 5 more zero-bias upper steps; `_calibrate_demand_from_radio_window()` fully converges

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Isolated-carrier version of the medium_270m topology.
# Each gNB gets a unique carrier_id so _same_carrier_interferers is empty
# and SINR = SNR (noise-limited).  A3 RSRP comparison is carrier-agnostic
# in this sim so handovers still fire normally.
_base = CENTER_GAP_GNB_CONFIGS['medium_270m']
GNB_CONFIGS = tuple(
    {**g, 'carrier_id': g['id']}   # gNB-0 → carrier 0, gNB-1 → 1, gNB-2 → 2
    for g in _base
)

GNB_POSITIONS = {g['id']: (g['x'], g['y']) for g in GNB_CONFIGS}
SLICE_COLORS  = {'eMBB': '#2196F3', 'URLLC': '#FF5722', 'mMTC': '#4CAF50'}
GNB_COLORS    = ['#1565C0', '#6A1B9A', '#1B5E20']
SCENARIO_NAMES = [
    'jain_balance_controllable',
    'jain_control_urllc',
    'jain_control_mmtc',
    'jain_control_mixed',
]
print('Ready — gNBs:', GNB_POSITIONS)
print('Carrier IDs:', {g['id']: g['carrier_id'] for g in GNB_CONFIGS})

In [ ]:
def gnb_total_useful_load(env):
    loads = env.base_env.get_slice_loads()
    return np.array([sum(loads.get((g, s), 0.0) for s in SLICE_TYPES) for g in range(3)])

def per_slice_load(env):
    loads = env.base_env.get_slice_loads()
    return {(g, s): loads.get((g, s), 0.0) for g in range(3) for s in SLICE_TYPES}

def jain(v):
    v = np.asarray(v, dtype=float)
    s2 = float(np.sum(v ** 2))
    return float(np.sum(v) ** 2) / (3 * s2) if s2 > 0 else 1.0

def ue_snapshot(env):
    return [{
        'x': float(ue.x), 'y': float(ue.y),
        'gnb': int(ue.serving_gnb),
        'slice': getattr(ue, 'slice_type', 'eMBB'),
        'demand_prbs': int(getattr(ue, 'upper_demand_prbs', 0)),
    } for ue in env.base_env.get_all_ues() if ue.connected and ue.serving_gnb is not None]

def make_env(scenario_name):
    return GlobalPPO3GNBEnv(
        seed=0,
        scenario_mode='curriculum',
        training_scenarios=scenario_name,
        gnb_configs=GNB_CONFIGS,       # isolated-carrier configs (no co-channel interference)
        global_steps_per_episode=12,
        local_steps_per_global=10,
        radio_substeps=20,
        warmup_steps=4,                # 4 steps → calibration converges before first measurement
        safe_admission_enabled=False,
        terminal_reward_only=False,
        post_handover_settle_steps=4,
    )

print('Helpers defined')

In [ ]:
results = {}

for name in SCENARIO_NAMES:
    env = make_env(name)
    env.reset()

    zero = np.zeros(env.action_space.shape, dtype=np.float32)
    neg  = np.full(env.action_space.shape, -1.0, dtype=np.float32)

    # State 1: warmup step with zero bias → initial load
    env.step(zero)
    sl_before  = per_slice_load(env)
    gl_before  = gnb_total_useful_load(env)
    ues_before = ue_snapshot(env)

    # State 2: full negative bias → handovers fire, first measurement window
    _, rew, _, _, info = env.step(neg)
    sl_transient  = per_slice_load(env)
    gl_transient  = gnb_total_useful_load(env)
    ues_after     = ue_snapshot(env)
    ho_count      = int(info.get('handover_count', 0))

    # State 3: 5 zero-bias steps → calibration fully converges (3-5 windows needed)
    for _ in range(5):
        env.step(zero)
    sl_settled = per_slice_load(env)
    gl_settled = gnb_total_useful_load(env)

    env.close()

    results[name] = {
        'ues_before'    : ues_before,
        'ues_after'     : ues_after,
        'sl_before'     : sl_before,
        'sl_transient'  : sl_transient,
        'sl_settled'    : sl_settled,
        'gl_before'     : gl_before,
        'gl_transient'  : gl_transient,
        'gl_settled'    : gl_settled,
        'jain_before'   : jain(gl_before),
        'jain_transient': jain(gl_transient),
        'jain_settled'  : jain(gl_settled),
        'ho_count'      : ho_count,
        'reward'        : float(rew),
    }

    print(f'{name:35s}  '
          f'J_before={results[name]["jain_before"]:.3f}  '
          f'J_transient={results[name]["jain_transient"]:.3f}  '
          f'J_settled={results[name]["jain_settled"]:.3f}  '
          f'ho={ho_count}')

---
## Spatial layout + PRB bars: before vs settled (post-calibration)

In [ ]:
SCENARIO_LABELS = {
    'jain_balance_controllable': 'eMBB only  (target 0.90)',
    'jain_control_urllc':        'URLLC only (target 0.72)',
    'jain_control_mmtc':         'mMTC only  (target 0.60)',
    'jain_control_mixed':        'Mixed 3-slice (total 0.90)',
}

fig, axes = plt.subplots(4, 3, figsize=(16, 18),
                         gridspec_kw={'width_ratios': [2, 1.4, 1.4]})

for row, name in enumerate(SCENARIO_NAMES):
    res = results[name]
    ax_map, ax_before, ax_after = axes[row]

    # ── spatial map ──────────────────────────────────────────────────────────
    for g, (gx, gy) in GNB_POSITIONS.items():
        ax_map.add_patch(plt.Circle((gx, gy), 500, color=GNB_COLORS[g], alpha=0.06))
        ax_map.plot(gx, gy, '^', ms=14, color=GNB_COLORS[g], zorder=5)
        ax_map.text(gx, gy + 545, f'gNB-{g}\n({gx:.0f} m)',
                    ha='center', va='bottom', fontsize=8, color=GNB_COLORS[g])

    # UEs before (faded)
    for ue in res['ues_before']:
        ax_map.scatter(ue['x'], ue['y'],
                       c=SLICE_COLORS.get(ue['slice'], 'gray'), s=50,
                       marker='o', alpha=0.30, zorder=3)

    # UEs after (bright, border = serving gNB)
    for ue in res['ues_after']:
        ax_map.scatter(ue['x'], ue['y'],
                       c=SLICE_COLORS.get(ue['slice'], 'gray'), s=80,
                       marker='o', edgecolors=GNB_COLORS[ue['gnb']],
                       linewidths=1.8, zorder=4)

    # Demand PRB labels on UEs (settled demand)
    for ue in res['ues_after']:
        ax_map.text(ue['x'], ue['y'] - 55, str(ue['demand_prbs']),
                    ha='center', va='top', fontsize=6.5, color='black', zorder=6)

    ax_map.set_xlim(-820, 820)
    ax_map.set_ylim(-640, 640)
    ax_map.set_aspect('equal')
    ax_map.set_title(f'{name}\n{SCENARIO_LABELS[name]}', fontsize=9, fontweight='bold')
    ax_map.set_xlabel('x (m)', fontsize=8)
    ax_map.set_ylabel('y (m)', fontsize=8)
    ax_map.tick_params(labelsize=7)
    ax_map.grid(alpha=0.20)

    # ── PRB load bars ─────────────────────────────────────────────────────────
    for ax, sl_dict, gl_vec, label, jain_val in [
        (ax_before, res['sl_before'],  res['gl_before'],  'BEFORE\n(all on gNB-1)', res['jain_before']),
        (ax_after,  res['sl_settled'], res['gl_settled'], f'SETTLED  ({res["ho_count"]} HOs)', res['jain_settled']),
    ]:
        x     = np.arange(3)
        w     = 0.25
        for s_idx, sl in enumerate(SLICE_TYPES):
            vals = np.array([sl_dict.get((g, sl), 0.0) for g in range(3)])
            ax.bar(x + (s_idx - 1) * w, vals, w,
                   label=sl, color=SLICE_COLORS[sl], alpha=0.85,
                   edgecolor='white', linewidth=0.5)

        # total useful load step line
        t = np.array([sum(sl_dict.get((g, s), 0.0) for s in SLICE_TYPES) for g in range(3)])
        for g_i, tv in enumerate(t):
            ax.plot([g_i - 0.42, g_i + 0.42], [tv, tv], 'k--', lw=1.5, alpha=0.7)
            ax.text(g_i, tv + 0.03, f'{tv:.2f}', ha='center', fontsize=7, color='black')

        ax.axhline(0.80, color='crimson', lw=1.2, linestyle=':', alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels([f'gNB-{g}' for g in range(3)], fontsize=8)
        ax.set_ylim(0, 1.20)
        ax.set_ylabel('useful PRB load', fontsize=7)
        ax.set_title(f'{label}\nJain = {jain_val:.3f}', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(axis='y', alpha=0.3)
        if row == 0:
            ax.legend(fontsize=7, loc='upper right')
            ax.axhline(0.80, color='crimson', lw=1.2, linestyle=':', alpha=0.7, label='safe limit (0.80)')
            ax.legend(fontsize=7, loc='upper right')

# Global legend
legend_els = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=SLICE_COLORS['eMBB'],   ms=9, label='eMBB UE'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=SLICE_COLORS['URLLC'],  ms=9, label='URLLC UE'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=SLICE_COLORS['mMTC'],   ms=9, label='mMTC UE'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', alpha=0.3, ms=9, label='before (faded)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
           markeredgecolor='black', ms=9, label='after (border = serving gNB)'),
    Line2D([0],[0], color='black', lw=1.5, linestyle='--', label='total useful load'),
    Line2D([0],[0], color='crimson', lw=1.2, linestyle=':', label='safe limit 0.80'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=4,
           fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.015))
fig.suptitle(
    'Jain Controllable Scenarios — Spatial Layout and Per-gNB Useful PRB Load\n'
    'UEs start on gNB-1 (center); numbers = demand_prbs per UE; bars = per-slice useful load fraction',
    fontsize=12, fontweight='bold', y=1.01,
)
fig.tight_layout()
plt.show()

---
## Post-HO calibration transient — load spike explained

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, name in zip(axes, SCENARIO_NAMES):
    res   = results[name]
    x     = np.arange(3)
    w     = 0.28
    states = [
        (res['gl_before'],   '#90CAF9', 'Before'),
        (res['gl_transient'],'#FFCC80', 'Post-HO\n(transient)'),
        (res['gl_settled'],  '#A5D6A7', 'Settled\n(5 steps)'),
    ]
    for i, (gl, color, label) in enumerate(states):
        bars = ax.bar(x + (i - 1) * w, gl, w, label=label,
                      color=color, edgecolor='gray', linewidth=0.6)
        for b in bars:
            if b.get_height() > 0.02:
                ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                        f'{b.get_height():.2f}', ha='center', fontsize=6)

    ax.axhline(0.80, color='crimson', lw=1.2, linestyle=':', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([f'gNB-{g}' for g in range(3)], fontsize=8)
    ax.set_ylim(0, 1.30)
    jb = res['jain_before']; jt = res['jain_transient']; js = res['jain_settled']
    ax.set_title(
        f'{name.replace("jain_","").replace("_"," ")}\n'
        f'J: {jb:.2f} → {jt:.2f} → {js:.2f}',
        fontsize=8,
    )
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(labelsize=7)

axes[0].set_ylabel('gnb_total_useful_load', fontsize=8)
axes[0].legend(fontsize=7)
fig.suptitle(
    'Post-handover calibration transient\n'
    'Outer gNBs show load spike immediately after HO;\n'
    '5 zero-bias upper steps let _calibrate_demand_from_radio_window() fully converge',
    fontsize=10,
)
fig.tight_layout()
plt.show()

---
## Summary table

In [ ]:
rows = []
for name in SCENARIO_NAMES:
    res = results[name]
    rows.append({
        'scenario'       : name.replace('jain_', ''),
        'gNB0_before'    : round(res['gl_before'][0],    3),
        'gNB1_before'    : round(res['gl_before'][1],    3),
        'gNB2_before'    : round(res['gl_before'][2],    3),
        'Jain_before'    : round(res['jain_before'],     3),
        'gNB0_transient' : round(res['gl_transient'][0], 3),
        'gNB1_transient' : round(res['gl_transient'][1], 3),
        'gNB2_transient' : round(res['gl_transient'][2], 3),
        'Jain_transient' : round(res['jain_transient'],  3),
        'gNB0_settled'   : round(res['gl_settled'][0],   3),
        'gNB1_settled'   : round(res['gl_settled'][1],   3),
        'gNB2_settled'   : round(res['gl_settled'][2],   3),
        'Jain_settled'   : round(res['jain_settled'],    3),
        'HOs'            : res['ho_count'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))